# Qwen2.5 Moroccan Law — Load from Drive Link & Test
No manual mounting needed. Just run cells top-to-bottom.

In [1]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU found. Runtime > Change runtime type > T4 GPU')


CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install -q gdown


In [3]:
import gdown, os

FOLDER_ID  = '189nkqTWAi0Mf620RfVL_eMnlU6GnqTPW'
LOCAL_DIR  = '/content/lora-adapter'

# Download all adapter files from the shared Drive folder
print('Downloading adapter from Drive...')
gdown.download_folder(
    id=FOLDER_ID,
    output=LOCAL_DIR,
    quiet=False,
    use_cookies=False,
)

# gdown sometimes nests an extra subfolder — find the real adapter root
def find_adapter_root(base):
    """Walk down single-child dirs until we find adapter_config.json."""
    for root, dirs, files in os.walk(base):
        if 'adapter_config.json' in files:
            return root
    return None

ADAPTER_PATH = find_adapter_root(LOCAL_DIR)
assert ADAPTER_PATH, (
    'adapter_config.json not found after download. '
    'Check that the Drive folder contains your LoRA adapter files.'
)

print('Adapter ready at:', ADAPTER_PATH)
print('Files:', os.listdir(ADAPTER_PATH))


Retrieving folder contents


Retrieving folder 19GVT-A4KiV1BWPLbeb1L-OcgJSLDpBrp checkpoints
Retrieving folder 1CuQ-EuZk1qQQnNST4UdI-sfJNDdfxvgW checkpoint-850
Processing file 11GLS8j5AADenGTlSE_4LvHhjtVxCecpU adapter_config.json
Processing file 1SpyrXhEVZLNkgFhoEDU6vHrPsJ_lkMgs adapter_model.safetensors
Processing file 1xzRAdx0QwA_NYDne1FRQ591kM0X01Toi chat_template.jinja
Processing file 19Dzd05tVpG1Ee1zDYJUUx7nblwP4tID2 optimizer.pt
Processing file 18MesBNXeMe2xyoIDu5o1KFQFWLjltKkn README.md
Processing file 1RLxNBpAtaxFGcgbbLnjD8qwN_AmScO1c rng_state.pth
Processing file 1NGL05cE_fkcrxZOcyhS_mL60DM3t-1h5 scaler.pt
Processing file 1S1ehnaJV9BXbmX_Z8Bmq-CzIE2_4ev_Z scheduler.pt
Processing file 1pSr96FOAgUXk8oS3oQdFWqRLVhR3e7AW tokenizer_config.json
Processing file 11g17uEjQSPYyXmG8f__9faowNEbO9YWF tokenizer.json
Processing file 1k7JTdOg0tW524K4pYx5-8l0LVUh7RRIp trainer_state.json
Processing file 10Hx5V4g6lUICqk7Wc3zYISUydFaG-AVx training_args.bin
Retrieving folder 1AJgSRrFvoT6pLp_cBT6Jgqtzd0Lcvx5I checkpoint-875
Pr

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=11GLS8j5AADenGTlSE_4LvHhjtVxCecpU
To: /content/lora-adapter/checkpoints/checkpoint-850/adapter_config.json
100%|██████████| 1.25k/1.25k [00:00<00:00, 4.45MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1SpyrXhEVZLNkgFhoEDU6vHrPsJ_lkMgs
From (redirected): https://drive.google.com/uc?id=1SpyrXhEVZLNkgFhoEDU6vHrPsJ_lkMgs&confirm=t&uuid=5313b217-647d-450b-8bd2-47ad619e85a0
To: /content/lora-adapter/checkpoints/checkpoint-850/adapter_model.safetensors
100%|██████████| 240M/240M [00:01<00:00, 131MB/s]  
Downloading...
From: https://drive.google.com/uc?id=1xzRAdx0QwA_NYDne1FRQ591kM0X01Toi
To: /content/lora-adapter/checkpoints/checkpoint-850/chat_template.jinja
100%|██████████| 2.51k/2.51k [00:00<00:00, 11.0MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=19Dzd05tVpG1Ee1zDYJUUx7nblwP4tID2
From (redi

Adapter ready at: /content/lora-adapter/lora-adapter
Files: ['tokenizer.json', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja', 'README.md', 'adapter_model.safetensors']



Download completed


In [4]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
print('Model loaded and ready.')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.2 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model loaded and ready.


## Retrieval-Augmented Generation (RAG) Setup

To enhance the Drive Model's ability to provide accurate and contextually relevant answers, we will integrate Retrieval-Augmented Generation (RAG). This involves:

1.  **Installing RAG Libraries:** Adding `langchain`, `faiss-cpu`, and `sentence-transformers`.
2.  **Loading Legal Documents:** Utilizing the legal documents previously downloaded to create a knowledge base.
3.  **Document Chunking:** Splitting the documents into smaller, manageable chunks.
4.  **Creating Embeddings:** Converting these chunks into numerical representations using an embedding model.
5.  **Building a Vector Store:** Storing these embeddings in a searchable database (FAISS).
6.  **Configuring a Retriever:** Setting up a mechanism to fetch relevant document chunks based on a user's query.


In [5]:
!pip install langchain-text-splitters langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, w

In [6]:
!pip install jq
!pip install langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 30.5 MB/s eta 0:00:00


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import json
import os

# Path to the cleaned legal documents
CLEANED_DATA_PATH = os.path.join(LOCAL_DIR, 'cleaned-data', 'moroccan_law_cleaned_all.jsonl')

# Ensure the file exists before proceeding
if not os.path.exists(CLEANED_DATA_PATH):
    raise FileNotFoundError(f"Cleaned data file not found at: {CLEANED_DATA_PATH}")

print(f"Loading documents from: {CLEANED_DATA_PATH}")

# Load documents manually (pas besoin de jq)
documents = []
with open(CLEANED_DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        
        page_content = f"Question: {record.get('input', '')}\nRéponse: {record.get('output', '')}"
        
        metadata = {
            "source":   record.get("source_line", ""),
            "language": record.get("language", ""),
            "type":     record.get("type", ""),
        }
        
        documents.append(Document(page_content=page_content, metadata=metadata))

print(f"Loaded {len(documents)} documents.")

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)
docs = text_splitter.split_documents(documents)
print(f"Split into {len(docs)} document chunks.")

Loading documents from: /content/lora-adapter/cleaned-data/moroccan_law_cleaned_all.jsonl
Loaded 1225 documents.
Split into 1229 document chunks.


In [10]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.3 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [11]:
# Initialize embedding model
# Using a Sentence-BERT model for embeddings
embedding_model_name = "intfloat/multilingual-e5-large"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cuda'}
)

print(f"Embedding model '{embedding_model_name}' loaded.")

# Create FAISS vector store
vector_store = FAISS.from_documents(docs, embeddings)
print("FAISS vector store created.")

# Create a retriever from the vector store
vector_store_retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Vector store retriever configured.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model 'intfloat/multilingual-e5-large' loaded.
FAISS vector store created.
Vector store retriever configured.


## Drive Model with RAG

Now, the `ask_model` function is re-defined to incorporate the RAG retriever. This function will now fetch relevant context for each question before passing it to the Qwen2.5 model.

In [ ]:
import torch

SYSTEM_PROMPT = """
You are a Moroccan legal assistant specialized ONLY in Moroccan law.

STRICT RULES:
- Answer ONLY questions related to Moroccan law and legal matters.
- If the question is not legal, refuse politely.
- Do NOT answer general knowledge, medical, political, religious, personal, or entertainment questions.
- Do NOT invent laws, article numbers, or legal sources.
- If the legal information is uncertain or missing from the context, say:
  "I do not have enough verified legal information to answer accurately."
- Use ONLY the provided legal context when available.
- If the answer is not present in the retrieved context, say so clearly.
- Always answer in the same language as the user.
- Keep answers concise, professional, and legally focused.
- Mention the law/article ONLY if explicitly present in the context.
- Never fabricate article numbers.
- Your responses are for legal information only and do not replace a lawyer.
"""

def ask_model(question: str,
              max_new_tokens: int = 180,
              temperature: float = 0.0,
              retriever=None) -> str:

    context = ""

    # =========================
    # 🔍 RAG RETRIEVAL
    # =========================
    if retriever:
        docs = retriever.invoke(question)

        # ❌ if no documents → stop immediately
        if not docs:
            return "Information not found in legal context."

        docs = docs[:4]

        context = "\n\nLEGAL CONTEXT:\n"
        context += "\n---\n".join([doc.page_content for doc in docs])

        # =========================
        # 🔥 DEBUG (VERY IMPORTANT)
        # =========================
        print("\n========== RETRIEVED CONTEXT ==========")
        print(context)
        print("=======================================\n")

    # =========================
    # PROMPT
    # =========================
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT + context
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt"
    ).to("cuda")

    # =========================
    # GENERATION (NO SAMPLING)
    # =========================
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return response


print("ask_model() ready with STRICT Moroccan legal RAG + DEBUG")

ask_model() ready with STRICT Moroccan legal RAG + DEBUG


## Drive Model Evaluation with RAG (Specialized Questions)

Now, let's evaluate the RAG-enabled Drive Model using the specialized legal questions. The `ask_model` function will now leverage the `vector_store_retriever` to provide contextually relevant answers.

In [15]:
import json, datetime

# Ensure 'questions' list is defined from previous cells (cell 7a63be9d)
# If running this cell out of order, you might need to re-run cell 7a63be9d.

if 'questions' not in globals():
    print("Warning: 'questions' list not found. Please ensure cell 7a63be9d or an equivalent cell defining 'questions' has been run.")
    questions = [] # Initialize an empty list to prevent further errors

print(f"Starting RAG-enabled Drive Model evaluation for {len(questions)} specialized questions...")

results_rag_specialized = []
for i, q in enumerate(questions, 1):
    print(f'\nProcessing question [{i}/{len(questions)}]')
    answer = ask_model(q, max_new_tokens=512, retriever=vector_store_retriever) # Use RAG
    results_rag_specialized.append({'question': q, 'answer': answer})
    print(f'Question: {q[:100]}...')
    print(f'Answer: {answer[:300]}...') # Print a snippet of the answer
    print('-' * 80)

# Save results as a timestamped JSON in /content
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
out_rag_specialized = f'/content/eval_drive_rag_specialized_{ts}.json'
with open(out_rag_specialized, 'w', encoding='utf-8') as f:
    json.dump(results_rag_specialized, f, ensure_ascii=False, indent=2)
print(f'\nDrive Model RAG results for specialized questions saved to: {out_rag_specialized}')

Starting RAG-enabled Drive Model evaluation for 0 specialized questions...

Drive Model RAG results for specialized questions saved to: /content/eval_drive_rag_specialized_20260517_205629.json


## Displaying RAG-enabled Drive Model Results

Here are the results from the Drive Model augmented with RAG for the specialized legal questions.

In [22]:
import pandas as pd
import json

# Load the RAG-enabled Drive model results for specialized questions
# Ensure 'out_rag_specialized' variable is available from the previous cell execution
if 'out_rag_specialized' not in locals():
    print("Error: 'out_rag_specialized' not found. Please ensure the previous RAG evaluation cell was run.")
else:
    with open(out_rag_specialized, 'r', encoding='utf-8') as f:
        drive_model_rag_results_df = json.load(f)

    # Create a DataFrame for displaying
    display_df = pd.DataFrame(drive_model_rag_results_df)

    # Display the results table
    pd.set_option('display.max_colwidth', None)
    display(display_df)

    print('\nRAG-enabled Drive Model results displayed successfully.')

""



RAG-enabled Drive Model results displayed successfully.


In [23]:
import json
import datetime

questions = [
    "What is the minimum age for criminal responsibility in Morocco?",
    "Quels sont les droits du locataire en cas d'expulsion au Maroc ?",
    "ما هي شروط الطلاق بالتراضي في القانون المغربي؟",
    "What documents are required to register a birth in Morocco?",
    "Quels sont les délais pour déclarer une naissance à l'état civil ?",
    "ما هي الوثائق المطلوبة لتسجيل عقد الزواج في الحالة المدنية؟",
    "Who is eligible for the family solidarity fund in Morocco?",
    "Quelles sont les conditions pour bénéficier du fonds de solidarité familiale ?",
    "كيف يمكن للمرأة المطلقة الاستفادة من صندوق التكافل العائلي؟",
    "Can a Moroccan woman transmit her nationality to her children?",
    "Quelles sont les conditions pour obtenir la nationalité marocaine par naturalisation ?",
    "ما هي حالات فقدان الجنسية المغربية؟",
    "What is the legal difference between kafala and adoption in Morocco?",
    "Quelles sont les conditions requises pour obtenir une kafala au Maroc ?",
    "ما هي حقوق الطفل المكفول في القانون المغربي؟",
    "What is the minimum legal age for marriage in Morocco?",
    "Quelles sont les causes légales du divorce au Maroc ?",
    "ما هي شروط تعدد الزوجات في مدونة الأسرة المغربية؟"
]

results = []

for i, q in enumerate(questions, 1):
    try:
        print(f"\n[{i}/{len(questions)}] {q}")

        answer = ask_model(q, max_new_tokens=180, retriever=vector_store_retriever)

        print("\nANSWER:")
        print(answer)
        print("-" * 80)

        results.append({
            "question": q,
            "answer": answer
        })

    except Exception as e:
        print("ERROR:", str(e))

        results.append({
            "question": q,
            "answer": None,
            "error": str(e)
        })

# save results
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
out = f'/content/eval_{ts}.json'

with open(out, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nResults saved to: {out}")

Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[1/18] What is the minimum age for criminal responsibility in Morocco?

========== RETRIEVED CONTEXT ==========


LEGAL CONTEXT:
Question: ما هو الحد الأدنى للسكن في المغرب بكيفية اعتيادية للاختيار الجنسية المغربية؟
Réponse: خمس عشرة سنة علي الاقل
---
Question: ما هو الحد الأدنى لممارسة وظيفة عمومية في الادارة المغربية للاختيار الجنسية المغربية؟
Réponse: عشر سنوات علي الاقل
---
Question: Answer this: Give me a medical diagnosis.
Réponse: I specialize only in Moroccan law. I cannot answer this question because it is outside Moroccan law.



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
the minimum age for criminal responsibility in morocco is 12 years old. **article 4**
--------------------------------------------------------------------------------

[2/18] Quels sont les droits du locataire en cas d'expulsion au Maroc ?

========== RETRIEVED CONTEXT ==========


LEGAL CONTEXT:
Question: Le kafil a-t-il le droit de voyager ou de s'installer définitivement à l'étranger avec l'enfant ?
Réponse: Selon l'article 21, le kafil ne peut quitter le territoire marocain pour s'installer définitivement à l'étranger avec l'enfant qu'après autorisation préalable du juge des tutelles et notification au consulat du Maroc du pays d'accueil.
---
Question: J'ai besoin d'aide : Que prévoit le droit français concernant le licenciement ?
Réponse: Je suis spécialisé uniquement dans le droit marocain. Je ne peux donc pas répondre à cette question, car elle ne relève pas du droit marocain.
---
Question: Quelles sont les conditions pour que les marocains résidant à l'étranger puissen

Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


ANSWER:
وفقاً للمادة 1 من القانون رقم 15.01، يعتبر الطفل المكيول شخصًا مهمّاً ويتمتع بحقوق خاصة، بما في ذلك الحق في العيش مع والديه، أو حق الاختيار عند بلوغه سن الرشد، أو حق الحصول على التعليم والformation التي تناسب عمره، كما يتمتع بالحق في عدم الإذلال أو الاستغلال.
--------------------------------------------------------------------------------

[16/18] What is the minimum legal age for marriage in Morocco?

========== RETRIEVED CONTEXT ==========


LEGAL CONTEXT:
Question: Quel est l'âge minimum pour se marier ?
Réponse: L'âge du mariage est fixé uniformément à 18 ans, mais le juge peut réduire cet âge dans les cas justifiés.
---
Question: ما هو الحد الأدنى للسكن في المغرب بكيفية اعتيادية للاختيار الجنسية المغربية؟
Réponse: خمس عشرة سنة علي الاقل
---
Question: Quelles sont les conditions pour qu'une femme étrangère acquière la nationalité marocaine par le mariage ?
Réponse: Selon l'article 10 du Code de la nationalité, elle doit justifier d'une résidence habituelle et régulière au 

Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
According to Article 4 of the code, the minimum legal age for marriage is 18 years old. However, the judge can reduce this age in justified cases.
--------------------------------------------------------------------------------

[17/18] Quelles sont les causes légales du divorce au Maroc ?

========== RETRIEVED CONTEXT ==========


LEGAL CONTEXT:
Question: Que prévoit le droit français concernant le licenciement ?
Réponse: Je suis spécialisé uniquement dans le droit marocain. Je ne peux donc pas répondre à cette question, car elle ne relève pas du droit marocain.
---
Question: Question : Que prévoit le droit français concernant le licenciement ?
Réponse: Je suis spécialisé uniquement dans le droit marocain. Je ne peux donc pas répondre à cette question, car elle ne relève pas du droit marocain.
---
Question: Réponds à ceci : Que prévoit le droit français concernant le licenciement ?
Réponse: Je suis spécialisé uniquement dans le droit marocain. Je ne peux donc pas répondre à c

Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
Selon l'article 71 du code de la famille marocain, les causes légales du divorce sont les suivantes : trompée par son conjoint (article 80), insubordination ou manquement aux devoirs conjugaux (article 82) et vice rédhibitoire (article 84).
--------------------------------------------------------------------------------

[18/18] ما هي شروط تعدد الزوجات في مدونة الأسرة المغربية؟

========== RETRIEVED CONTEXT ==========


LEGAL CONTEXT:
Question: من يخضع لأحكام مدونة الأسرة؟
Réponse: جميع المغاربة ولو كانوا حاملين لجنسية أخرى
---
Question: مغربي يحمل جنسية أخرى، هل تسري عليه أحكام مدونة الأسرة؟
Réponse: وفق المادة 2، تسري أحكام هذه المدونة على جميع المغاربة ولو كانوا حاملين لجنسية أخرى
---
Question: ما هي شروط قبول الحكم الأجنبي بالطلاق في المغرب؟
Réponse: وفق المادة 128، الحكم الأجنبي بالطلاق يكون قابلاً للتنفيذ إذا صدر عن محكمة مختصة وأستند إلى أسباب لا تتنافى مع التي قررتها مدونة الأسرة


ANSWER:
وفق المادة 36، يمكن للمواطن أن يتزوج بثلاث زوجات فقط، ولكن يجب أن تكون لديه القد

## Side-by-Side Comparison of Model Outputs (Extended Questions)

In [24]:
import pandas as pd
import json

# 1. Évaluation de la Baseline (Sans RAG) sur le GPU
print('Evaluating Baseline (No-RAG)...')
baseline_results = []
for i, q in enumerate(questions, 1):
    print(f'[{i}/{len(questions)}] {q[:60]}...')
    ans = ask_model(q, max_new_tokens=180, retriever=None) # No RAG
    baseline_results.append({'question': q, 'answer': ans})

# 2. Alignement des résultats RAG et Sans RAG
drive_questions = [r['question'] for r in results]
drive_answers = [r['answer'] for r in results] # Avec RAG (Cellule 16)
baseline_answers = [r['answer'] for r in baseline_results] # Sans RAG (Cette cellule)

# 3. Création du DataFrame pour la comparaison
comparison_df = pd.DataFrame({
    'Question': drive_questions,
    'Fine-Tuned + RAG Model (Avec RAG)': drive_answers,
    'Fine-Tuned Baseline (Sans RAG)': baseline_answers
})

# Affichage du tableau interactif
pd.set_option('display.max_colwidth', None)
display(comparison_df)

print('\nComparison table generated successfully.')


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating Baseline (No-RAG)...
[1/18] What is the minimum age for criminal responsibility in Moroc...


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/18] Quels sont les droits du locataire en cas d'expulsion au Mar...


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[17/18] Quelles sont les causes légales du divorce au Maroc ?...


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[18/18] ما هي شروط تعدد الزوجات في مدونة الأسرة المغربية؟...


,Question,Fine-Tuned + RAG Model (Avec RAG),Fine-Tuned Baseline (Sans RAG)
0,What is the minimum age for criminal responsibility in Morocco?,the minimum age for criminal responsibility in morocco is 12 years old. **article 4**,"According to Article 4 of the Code pénal marocain, an individual is criminally responsible at the age of 12 years. This means that children under the age of 12 are considered minors and may be exempt from certain penalties for their actions. The reasoning behind this rule is to protect children from harsh punishments and ensure they receive guidance and support during their formative years."
1,Quels sont les droits du locataire en cas d'expulsion au Maroc ?,"Selon l'article 36 bis, le locataire bénéficie d'un délai maximum de deux mois avant son expulsion, pendant lequel il doit être informé par écrit de la décision prise contre lui. Il est également autorisé à présenter une demande de rectification de fautes ou d'erreurs qui ont entraîné sa condamnation.","selon l'article 18 de la loi n° 37-99 relative à l'état civil, le locataire peut bénéficier des prestations du fonds d'entraide familiale si son éviction entraîne une perte substantielle de revenus."
2,ما هي شروط الطلاق بالتراضي في القانون المغربي؟,وفقا للمادة 96، يجب أن يتوافق التراضي مع القواعد الشرعية وأن يتم إثباته من قبل الزوجين مباشرة أو من خلال شاهدين.,الطلاق بالتراضي هو طريقة لانفصال الزوجين، ويتم ذلك بعد أن يتوافق الزوجان على الطلاق من خلال اتفاق مشترك. يجب أن يكون هذا الاتفاق مكتوباً وواضحاً، وأن يتم إيداعه في السجلات الرسمية.
3,What documents are required to register a birth in Morocco?,"According to Article 2 of the Law No. 37-99 on Civil Status, the following documents must be presented: an attestation of marriage (for both parents), proof of death of the mother within three months before registration, and a copy of the child's identity card. Additionally, there may be other requirements depending on the specific circumstances.","According to Article 16 of the Civil Registration Code, the following documents must be submitted: (1) an excerpt of the marriage certificate of the parents; (2) proof of death of the mother within three months prior to registration. These documents serve as essential evidence to validate the birth declaration process."
4,Quels sont les délais pour déclarer une naissance à l'état civil ?,"Selon l'article 16, la déclaration de naissance doit être faite dans un délai de trente jours à compter de la date de l'accouchement. Si cela n'est pas fait, il faut demander une autorisation spéciale au procureur du roi.","selon l'article 31 du code de la famille marocain, le délai est de trois mois à compter de la date d'arrivée ou de décès."
5,ما هي الوثائق المطلوبة لتسجيل عقد الزواج في الحالة المدنية؟,وفقا للمادة 19، يجب تقديم وثائق مثل شهادة زفاف وشهادة تأكيد الزواج وشهادة ولادة للزوجين.,يجب تقديم وثائق معينة، بما ي包括: نسخة من عقد الزواج، ونسخة من شهادة الميلاد لكل الزوجين، وشهادة إدارية للزوجين.
6,Who is eligible for the family solidarity fund in Morocco?,"According to Article 2, individuals who are Moroccans of any nationality (including those with another nationality) and refugees/apatrids are eligible for the family solidarity fund. This includes families requiring financial assistance due to various circumstances such as divorce, death of the spouse, etc.","according to the decree n° 2-10-379 of july 8, 2010, families who meet certain conditions can benefit from the prestations du fonds d'entraide familiale. these conditions include being entitled to one of the following benefits: pension alimentaire due à l'épouse au marocain marié(e) avec enfant·e(s), allocation familiale due aux parents de l'enfant soumis à la kafala ou une aide spécifique pour les enfants handicapés. (reasoning: this provision outlines the eligibility criteria for receiving funds from the family solidarity fund.) **article 4**"
7,Quelles sont les conditions pour bénéficier du fonds de solidarité familiale ?,"selon l'article 120 du texte, il faut a


Comparison table generated successfully.


### Re-evaluating the Drive Model with all questions

In [ ]:
import json, datetime

results = []

for i, q in enumerate(questions, 1):

    answer = ask_model(q, max_new_tokens=512)

    results.append({
        "question": q,
        "answer": answer
    })

    print(f"[{i}/{len(questions)}] {q[:70]}")
    print(answer)
    print("-" * 80)

# Save results as timestamped JSON in /content
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
out = f'/content/eval_{ts}.json'

with open(out, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'Drive Model results saved to: {out}')

## Updated Side-by-Side Comparison of Model Outputs

In [ ]:
import pandas as pd
import json

# 1. Run Baseline (Without RAG) on the T4 GPU with 512 tokens
print('Evaluating Baseline (No-RAG, 512 tokens)...')
baseline_results = []
for i, q in enumerate(questions, 1):
    print(f'[{i}/{len(questions)}] {q[:60]}...')
    ans = ask_model(q, max_new_tokens=512, retriever=None) # No RAG
    baseline_results.append({'question': q, 'answer': ans})

# 2. Align with RAG results from Cell 30 (saved in 'results')
drive_questions = [r['question'] for r in results]
drive_answers = [r['answer'] for r in results]
baseline_answers = [r['answer'] for r in baseline_results]

# 3. Create a DataFrame for comparison
comparison_df = pd.DataFrame({
    'Question': drive_questions,
    'Fine-Tuned + RAG Model': drive_answers,
    'Fine-Tuned (No RAG) Baseline': baseline_answers
})

# Display the comparison table
pd.set_option('display.max_colwidth', None)
display(comparison_df)

print('\nComparison table generated successfully.')
